# V12: Diversity via SEED=123

**V11 Copy with Different Fold Splits**

Identical to V11 (SEED=42) but with SEED=123 for different fold randomization.

- Same feature engineering: 249 features (V11)
- Same TE strategy: 16 base × 3 stats + 4 trigram ×1 = 52 TE features per fold
- XGB params from bi/tri notebook (lr=0.0063)
- LGBM + CatBoost: V6 best params
- 20-fold CV (no refit)
- Expected CV: ~0.9180 (similar to V11)
- Expected LB: ~0.9157+ (cross-blend with V32 should exceed 0.91572)

**Runtime**: ~10h | **Budget**: 13h

## 1. Libraries & Setup

In [ ]:
import numpy as np
import pandas as pd
import itertools, os, warnings, gc
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

from scipy.stats import rankdata

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb_lib

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED    = 123   # ← ONLY CHANGE from V11 (was 42)
N_FOLDS = 20
TRES    = 0.999
ES_XGB  = 500
ES_OTHER = 200
np.random.seed(SEED)
print(f'SEED={SEED} (V11 was 42 - different fold splits for diversity)')
print('All libraries loaded!')

## 2. Load All Datasets

In [ ]:
DATASET  = '/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset'
train    = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test     = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
original = pd.read_csv(f'{DATASET}/Original.csv')

print(f'Train:    {train.shape}')
print(f'Test:     {test.shape}')
print(f'Original: {original.shape}')

original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
original['TotalCharges'].fillna(original['TotalCharges'].median(), inplace=True)
original['Churn_bin'] = (original['Churn'] == 'Yes').astype(int)
original['id'] = range(900000, 900000 + len(original))

train_combined = pd.concat([
    train,
    original.drop(columns=['customerID'], errors='ignore')
], ignore_index=True)
print(f'Combined train: {train_combined.shape}')

v18_sub = pd.read_csv(f'{DATASET}/V18.csv')
v29_sub = pd.read_csv(f'{DATASET}/V29.csv')
v32_sub = pd.read_csv(f'{DATASET}/V32.csv')
print(f'V18 loaded (LB=0.91422): {v18_sub.shape}')
print(f'V29 loaded (LB=0.91408, CV=0.91643): {v29_sub.shape}')
print(f'V32 loaded (LB=0.91572 BEST, CV=0.91808): {v32_sub.shape}')

## 3. Pre-compute Static Feature Maps

In [ ]:
CATS_ORIG = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
NUMS_ORIG = ['tenure', 'MonthlyCharges', 'TotalCharges']

global_churn_mean = original['Churn_bin'].mean()
k = 10

orig_te = {}
for col in CATS_ORIG + NUMS_ORIG:
    if col in original.columns:
        stats = original.groupby(col)['Churn_bin'].agg(['mean', 'count'])
        stats['smoothed'] = (
            (stats['mean'] * stats['count'] + global_churn_mean * k)
            / (stats['count'] + k)
        )
        orig_te[col] = stats['smoothed'].to_dict()

orig_stats_maps = {}
for col in CATS_ORIG:
    if col in original.columns:
        s = original.groupby(col)['Churn_bin'].agg(['std', 'count']).reset_index()
        s.columns = [col, f'ORIG_std_{col}', f'ORIG_cnt_{col}']
        s[f'ORIG_std_{col}'] = s[f'ORIG_std_{col}'].fillna(0)
        s[f'ORIG_cnt_norm_{col}'] = s[f'ORIG_cnt_{col}'] / s[f'ORIG_cnt_{col}'].max()
        orig_stats_maps[col] = s.drop(columns=[f'ORIG_cnt_{col}']).set_index(col)

TOP15_BIGRAM_PAIRS = [
    ('Contract', 'InternetService'), ('Contract', 'PaymentMethod'),
    ('Contract', 'OnlineSecurity'),  ('Contract', 'TechSupport'),
    ('Contract', 'PaperlessBilling'),('InternetService', 'PaymentMethod'),
    ('InternetService', 'OnlineSecurity'), ('InternetService', 'TechSupport'),
    ('InternetService', 'StreamingTV'),    ('InternetService', 'StreamingMovies'),
    ('PaymentMethod', 'PaperlessBilling'), ('SeniorCitizen', 'Contract'),
    ('SeniorCitizen', 'InternetService'),  ('OnlineSecurity', 'TechSupport'),
    ('StreamingTV', 'StreamingMovies'),
]
bigram_orig_maps = {}
for c1, c2 in TOP15_BIGRAM_PAIRS:
    if c1 in original.columns and c2 in original.columns:
        combo_map = original.copy()
        combo_map['_key'] = combo_map[c1].astype(str) + '_' + combo_map[c2].astype(str)
        rate_map = combo_map.groupby('_key')['Churn_bin'].agg(['mean','count']).reset_index()
        rate_map.columns = ['_key','rate','cnt']
        rate_map['smoothed'] = (
            (rate_map['rate'] * rate_map['cnt'] + global_churn_mean * k)
            / (rate_map['cnt'] + k)
        )
        bigram_orig_maps[(c1,c2)] = rate_map.set_index('_key')['smoothed'].to_dict()

orig_churner_tc    = original.loc[original['Churn_bin']==1, 'TotalCharges'].values
orig_nonchurner_tc = original.loc[original['Churn_bin']==0, 'TotalCharges'].values
orig_tc            = original['TotalCharges'].values
orig_is_mc_mean    = original.groupby('InternetService')['MonthlyCharges'].mean()

def pctrank_against(values, reference):
    ref_sorted = np.sort(reference)
    return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

def zscore_against(values, reference):
    mu, sigma = np.mean(reference), np.std(reference)
    if sigma == 0: return np.zeros(len(values), dtype='float32')
    return ((values - mu) / sigma).astype('float32')

ch_q25  = np.quantile(orig_churner_tc, 0.25)
ch_q50  = np.quantile(orig_churner_tc, 0.50)
ch_q75  = np.quantile(orig_churner_tc, 0.75)
nc_q25  = np.quantile(orig_nonchurner_tc, 0.25)
nc_q50  = np.quantile(orig_nonchurner_tc, 0.50)
nc_q75  = np.quantile(orig_nonchurner_tc, 0.75)

freq_maps_num = {}
for col in NUMS_ORIG:
    combined_vals = pd.concat([train_combined[col], test[col]])
    freq_maps_num[col] = combined_vals.value_counts(normalize=True).to_dict()

print('Building OrdinalEncoder...')
all_cat_data = pd.concat([
    train_combined[CATS_ORIG].fillna('MISSING'),
    test[CATS_ORIG].fillna('MISSING')
], axis=0)
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
oe.fit(all_cat_data)

TOP4_FOR_TRIGRAM = ['Contract', 'InternetService', 'PaymentMethod', 'OnlineSecurity']
TRIGRAM_PAIRS = list(itertools.combinations(TOP4_FOR_TRIGRAM, 3))
print(f'Trigram pairs: {len(TRIGRAM_PAIRS)}')

def add_le_cols(train_df, test_df):
    train_df, test_df = train_df.copy(), test_df.copy()
    n_train = len(train_df)
    tr_le = oe.transform(train_df[CATS_ORIG].fillna('MISSING')).astype(np.int32)
    te_le = oe.transform(test_df[CATS_ORIG].fillna('MISSING')).astype(np.int32)
    for i, col in enumerate(CATS_ORIG):
        train_df[f'{col}_le'] = tr_le[:, i]
        test_df[f'{col}_le']  = te_le[:, i]
    for c1, c2, c3 in TRIGRAM_PAIRS:
        tr_str   = train_df[c1].astype(str) + '_' + train_df[c2].astype(str) + '_' + train_df[c3].astype(str)
        te_str   = test_df[c1].astype(str)  + '_' + test_df[c2].astype(str)  + '_' + test_df[c3].astype(str)
        combined = pd.concat([tr_str, te_str], axis=0).reset_index(drop=True)
        codes    = pd.factorize(combined)[0]
        col_name = f'TG_{c1}_{c2}_{c3}_le'
        train_df[col_name] = codes[:n_train]
        test_df[col_name]  = codes[n_train:]
    return train_df, test_df

train_combined, test = add_le_cols(train_combined, test)
print(f'ORIG_proba: {len(orig_te)}, ORIG_stats: {len(orig_stats_maps)*2}')
print(f'Bigram ORIG-rate: {len(bigram_orig_maps)}, Trigram LE: {len(TRIGRAM_PAIRS)}')

## 4. Preprocessing (Identical to V11)

In [ ]:
def preprocess(df, is_train=True, freq_maps=None, fit_freq=False):
    df = df.copy()
    df.drop(columns=['id', 'customerID'], inplace=True, errors='ignore')
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
    if is_train and 'Churn' in df.columns:
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    df.drop(columns=['Churn_bin'], inplace=True, errors='ignore')

    for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
        if col in df.columns:
            df[col] = (df[col] == 'Yes').astype(int)
    if 'gender' in df.columns:
        df['gender'] = (df['gender'] == 'Male').astype(int)

    df['AvgMonthlyCharge']  = df['TotalCharges'] / df['tenure'].clip(lower=1)
    df['ChargeRatio']       = df['MonthlyCharges'] / df['AvgMonthlyCharge'].clip(lower=0.01)
    df['ChargeXTenure']     = df['MonthlyCharges'] * df['tenure']
    df['LifetimeValue']     = df['TotalCharges'] / df['MonthlyCharges'].clip(lower=0.01)
    df['ChargeGrowth']      = df['MonthlyCharges'] - df['AvgMonthlyCharge']
    df['ChargeGrowthPct']   = df['ChargeGrowth'] / df['AvgMonthlyCharge'].clip(lower=0.01)
    df['ChargePerMonth']    = df['MonthlyCharges'] / df['tenure'].clip(lower=1)
    df['IsNewCustomer']     = (df['tenure'] <= 12).astype(int)
    df['IsLongTerm']        = (df['tenure'] >= 60).astype(int)
    df['IsHighValue']       = (df['MonthlyCharges'] > 70).astype(int)

    if 'Contract' in df.columns:
        cr_map = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
        df['ContractRisk']       = df['Contract'].map(cr_map).fillna(2)
        df['TenureContractRisk'] = df['tenure'] * df['ContractRisk']
        df['ChargeContractRisk'] = df['MonthlyCharges'] * df['ContractRisk']
        df['HighRiskFlag']       = ((df['ContractRisk']==3) & (df['IsNewCustomer']==1)).astype(int)

    service_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                    'TechSupport','StreamingTV','StreamingMovies']
    temp_bins = []
    for col in service_cols:
        if col in df.columns and df[col].dtype == object:
            bname = col + '_bin'
            df[bname] = (df[col] == 'Yes').astype(int)
            temp_bins.append(bname)
    df['TotalServices']    = df['PhoneService'].copy()
    for bname in temp_bins:
        df['TotalServices'] += df[bname]
    df['ChargePerService'] = df['MonthlyCharges'] / df['TotalServices'].clip(lower=1)
    df['NoProtection']     = sum(
        (df[c] == 'No').astype(int) for c in ['OnlineSecurity','OnlineBackup','TechSupport']
        if c in df.columns
    )
    if 'PaymentMethod' in df.columns:
        df['AutoPay']         = df['PaymentMethod'].str.contains('automatic',case=False,na=False).astype(int)
        df['ElectronicCheck'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    if 'InternetService' in df.columns:
        df['FiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
        df['NoInternet'] = (df['InternetService'] == 'No').astype(int)
    if 'SeniorCitizen' in df.columns:
        df['SeniorXFiber']   = df['SeniorCitizen'] * df.get('FiberOptic', 0)
        df['SeniorXMonthly'] = df['SeniorCitizen'] * df['MonthlyCharges']
    df['IsStreamer']      = ((df.get('StreamingTV_bin',0)==1) & (df.get('StreamingMovies_bin',0)==1)).astype(int)
    df['HasFamily']       = ((df.get('Partner',0)==1) | (df.get('Dependents',0)==1)).astype(int)
    df['LonelyHighValue'] = ((df['HasFamily']==0) & (df['MonthlyCharges']>70)).astype(int)
    no_addon_flags = [(df[c]=='No').astype(int) for c in
                      ['OnlineSecurity','OnlineBackup','TechSupport','DeviceProtection'] if c in df.columns]
    if no_addon_flags:
        df['NoAddOns'] = (pd.concat(no_addon_flags, axis=1).sum(axis=1)==len(no_addon_flags)).astype(int)
    if 'ContractRisk' in df.columns:
        df['ContractXFiber']  = df['ContractRisk'] * df.get('FiberOptic', 0)
        df['MTMFiber']        = ((df['ContractRisk']==3) & (df.get('FiberOptic',0)==1)).astype(int)
        df['ServicesXRisk']   = df['TotalServices'] * df['ContractRisk']
        df['ExtremeRisk']     = (
            (df['IsNewCustomer']==1) & (df['IsHighValue']==1) & (df['ContractRisk']==3)
        ).astype(int)
    if 'PaperlessBilling' in df.columns and 'ElectronicCheck' in df.columns:
        df['PaperlessXElec'] = df['PaperlessBilling'] * df['ElectronicCheck']
    df['LoyalCustomer'] = ((df['IsLongTerm']==1) & (df.get('AutoPay',0)==1)).astype(int)
    df['tenure_sq']   = df['tenure'] ** 2
    df['monthly_sq']  = df['MonthlyCharges'] ** 2
    df['tenure_sqrt'] = np.sqrt(df['tenure'])
    df.drop(columns=temp_bins, inplace=True, errors='ignore')

    for col in NUMS_ORIG:
        df[f'FREQ_{col}'] = df[col].map(freq_maps_num[col]).fillna(0).astype('float32')

    t_str = df['tenure'].astype(str)
    df['tenure_first_digit']    = t_str.str[0].astype(int)
    df['tenure_last_digit']     = t_str.str[-1].astype(int)
    df['tenure_mod10']          = df['tenure'] % 10
    df['tenure_mod12']          = df['tenure'] % 12
    df['tenure_is_round10']     = (df['tenure'] % 10 == 0).astype('float32')
    df['tenure_dev_round10']    = np.abs(df['tenure'] - np.round(df['tenure']/10)*10)
    df['tenure_years']          = df['tenure'] // 12
    df['tenure_months_in_year'] = df['tenure'] % 12
    mc_fl = np.floor(df['MonthlyCharges'])
    df['mc_first_digit']  = df['MonthlyCharges'].astype(str).str.replace('.','',regex=False).str[0].astype(int)
    df['mc_mod10']        = mc_fl % 10
    df['mc_mod100']       = mc_fl % 100
    df['mc_is_round10']   = (mc_fl % 10 == 0).astype('float32')
    df['mc_fractional']   = df['MonthlyCharges'] - mc_fl
    df['mc_dev_round10']  = np.abs(df['MonthlyCharges'] - np.round(df['MonthlyCharges']/10)*10)
    tc_fl = np.floor(df['TotalCharges'])
    df['tc_first_digit']  = df['TotalCharges'].astype(str).str.replace('.','',regex=False).str[0].astype(int)
    df['tc_mod10']        = tc_fl % 10
    df['tc_mod100']       = tc_fl % 100
    df['tc_is_round100']  = (tc_fl % 100 == 0).astype('float32')
    df['tc_fractional']   = df['TotalCharges'] - tc_fl
    df['tc_dev_round100'] = np.abs(df['TotalCharges'] - np.round(df['TotalCharges']/100)*100)

    tc = df['TotalCharges'].values
    df['pctrank_churner_TC']    = pctrank_against(tc, orig_churner_tc)
    df['pctrank_nonchurner_TC'] = pctrank_against(tc, orig_nonchurner_tc)
    df['pctrank_orig_TC']       = pctrank_against(tc, orig_tc)
    df['pctrank_churn_gap_TC']  = (pctrank_against(tc, orig_churner_tc) -
                                   pctrank_against(tc, orig_nonchurner_tc)).astype('float32')
    df['zscore_churner_TC']     = zscore_against(tc, orig_churner_tc)
    df['zscore_nonchurner_TC']  = zscore_against(tc, orig_nonchurner_tc)
    df['zscore_churn_gap_TC']   = (np.abs(zscore_against(tc, orig_churner_tc)) -
                                   np.abs(zscore_against(tc, orig_nonchurner_tc))).astype('float32')
    df['resid_IS_MC']           = (df['MonthlyCharges'] -
                                   df['InternetService'].map(orig_is_mc_mean).fillna(0)).astype('float32')
    vals = np.zeros(len(df), dtype='float32')
    for cat_val in original['InternetService'].unique():
        mask = df['InternetService'] == cat_val
        ref  = original.loc[original['InternetService']==cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask,'TotalCharges'].values, ref)
    df['cond_pctrank_IS_TC'] = vals
    vals = np.zeros(len(df), dtype='float32')
    for cat_val in original['Contract'].unique():
        mask = df['Contract'] == cat_val
        ref  = original.loc[original['Contract']==cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask,'TotalCharges'].values, ref)
    df['cond_pctrank_C_TC'] = vals

    tc = df['TotalCharges'].values
    for q_label, ch_q, nc_q in [('q25',ch_q25,nc_q25),('q50',ch_q50,nc_q50),('q75',ch_q75,nc_q75)]:
        df[f'dist_ch_{q_label}']   = np.abs(tc - ch_q).astype('float32')
        df[f'dist_nc_{q_label}']   = np.abs(tc - nc_q).astype('float32')
        df[f'qdist_gap_{q_label}'] = (np.abs(tc - nc_q) - np.abs(tc - ch_q)).astype('float32')

    for col, te_dict in orig_te.items():
        if col in df.columns:
            fallback = float(np.mean(list(te_dict.values())))
            df[f'ORIG_proba_{col}'] = df[col].map(te_dict).fillna(fallback)
    for col, stats_df in orig_stats_maps.items():
        if col in df.columns:
            for c in stats_df.columns:
                df[c] = df[col].map(stats_df[c])
                df[c] = df[c].fillna(stats_df[c].mean())
    for (c1,c2), rate_map in bigram_orig_maps.items():
        if c1 in df.columns and c2 in df.columns:
            key = df[c1].astype(str) + '_' + df[c2].astype(str)
            df[f'BIGRAM_ORIG_{c1}_{c2}'] = key.map(rate_map).fillna(global_churn_mean)

    combos = [
        ('Contract','InternetService'),('Contract','PaymentMethod'),
        ('Contract','OnlineSecurity'),('Contract','TechSupport'),
        ('InternetService','PaymentMethod'),('InternetService','OnlineSecurity'),
        ('InternetService','TechSupport'),('PaymentMethod','PaperlessBilling'),
        ('MultipleLines','InternetService'),('StreamingTV','StreamingMovies'),
        ('OnlineSecurity','TechSupport'),('DeviceProtection','OnlineBackup'),
        ('Contract','PaperlessBilling'),('Contract','MultipleLines'),
        ('Contract','StreamingTV'),('InternetService','StreamingTV'),
        ('InternetService','StreamingMovies'),('InternetService','DeviceProtection'),
        ('SeniorCitizen','Contract'),('SeniorCitizen','InternetService'),
        ('SeniorCitizen','PaymentMethod'),('Partner','Contract'),
        ('Dependents','Contract'),('PhoneService','MultipleLines'),
        ('OnlineBackup','TechSupport'),('TechSupport','StreamingMovies'),
        ('Contract','DeviceProtection'),('gender','Contract'),
    ]
    for c1, c2 in combos:
        if c1 in df.columns and c2 in df.columns:
            combo_key = df[c1].astype(str) + '_' + df[c2].astype(str)
            if c1 in original.columns and c2 in original.columns:
                combo_map = original.copy()
                combo_map['_key'] = combo_map[c1].astype(str)+'_'+combo_map[c2].astype(str)
                rate_map = combo_map.groupby('_key')['Churn_bin'].mean().to_dict()
                df[f'COMBO_{c1}_{c2}'] = combo_key.map(rate_map).fillna(global_churn_mean)
            else:
                df[f'COMBO_{c1}_{c2}'] = combo_key.astype('category').cat.codes

    for col in ['tenure','MonthlyCharges','TotalCharges']:
        if col in df.columns:
            df[f'{col}_qbin'] = pd.qcut(df[col],q=10,labels=False,duplicates='drop').astype(float).fillna(0)
            df[f'{col}_ebin'] = pd.cut(df[col],bins=10,labels=False).astype(float).fillna(0)

    df['TenureGroup'] = pd.cut(
        df['tenure'],bins=[0,12,24,36,48,60,72],labels=[1,2,3,4,5,6]
    ).astype(float).fillna(1)

    cat_freq_cols = [
        'MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
        'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
        'Contract','PaymentMethod'
    ]
    cat_freq_cols = [c for c in cat_freq_cols if c in df.columns]
    if fit_freq:
        freq_maps = {col: df[col].value_counts().to_dict() for col in cat_freq_cols}
    if freq_maps:
        for col in cat_freq_cols:
            if col in freq_maps:
                df[f'{col}_freq'] = df[col].map(freq_maps[col]).fillna(0)

    ohe_cols = [
        'MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
        'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
        'Contract','PaymentMethod'
    ]
    ohe_cols = [c for c in ohe_cols if c in df.columns]
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=int)
    return df, freq_maps


train_clean, freq_maps = preprocess(train_combined, is_train=True, fit_freq=True)
test_clean,  _         = preprocess(test, is_train=False, freq_maps=freq_maps)
feature_cols = [c for c in train_clean.columns if c != 'Churn']
for col in feature_cols:
    if col not in test_clean.columns:
        test_clean[col] = 0
test_clean = test_clean[feature_cols]
print(f'Train clean: {train_clean.shape}, Test clean: {test_clean.shape}')

## 5. Setup & Parameters

In [ ]:
y   = train_clean['Churn']
X   = train_clean.drop(columns=['Churn'])
y_arr  = y.values
test_X = test_clean.copy()

BASE_LE_COLS    = [c for c in X.columns if c.endswith('_le') and not c.startswith('TG_')]
TRIGRAM_LE_COLS = [c for c in X.columns if c.startswith('TG_') and c.endswith('_le')]
ALL_LE_COLS     = BASE_LE_COLS + TRIGRAM_LE_COLS

cv               = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
churn_ratio      = y.mean()
scale_pos_weight = (1 - churn_ratio) / churn_ratio

print(f'Features: {X.shape[1]}, Train rows: {len(X):,}, SEED={SEED}')
print(f'TE per fold: {len(BASE_LE_COLS)*3 + len(TRIGRAM_LE_COLS)} features')

try:
    from cuml.preprocessing import TargetEncoder as cumlTE
    USE_CUML = True
    print('cuML: AVAILABLE')
except ImportError:
    USE_CUML = False

# Identical params to V11
best_xgb_params = {
    'n_estimators': 50000, 'learning_rate': 0.0063, 'max_depth': 5,
    'subsample': 0.81, 'colsample_bytree': 0.32, 'min_child_weight': 6,
    'reg_alpha': 3.5017, 'reg_lambda': 1.2925, 'gamma': 0.790,
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'early_stopping_rounds': ES_XGB, 'enable_categorical': True,
    'device': 'cuda', 'verbosity': 0, 'random_state': SEED,
}
best_lgbm_params = {
    'max_depth': 7, 'learning_rate': 0.02706, 'num_leaves': 55,
    'min_child_samples': 71, 'subsample': 0.8215, 'bagging_freq': 1,
    'colsample_bytree': 0.7317, 'reg_alpha': 7.34, 'reg_lambda': 0.000117,
}
best_cat_params = {
    'depth': 4, 'learning_rate': 0.13171, 'l2_leaf_reg': 4.283,
    'bagging_temperature': 0.9519, 'random_strength': 2.207,
    'border_count': 253, 'min_data_in_leaf': 33,
}
print('Params: identical to V11')

## 6. OOF - 20 Folds + Inner-Fold TE + Pseudo Labels

In [ ]:
oof_xgb  = np.zeros(len(X))
oof_lgbm = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_xgb_oof  = np.zeros(len(test_X))
test_lgbm_oof = np.zeros(len(test_X))
test_cat_oof  = np.zeros(len(test_X))

fold_scores = {'XGB': [], 'LGBM': [], 'CAT': []}
pl_counts   = {'XGB': 0, 'LGBM': 0, 'CAT': 0}

TE_STATS_BASE    = ['mean', 'var', 'median']
TE_STATS_TRIGRAM = ['mean']

print(f'OOF — {N_FOLDS} folds, SEED={SEED}, 3 models + TE + pseudo labels')
print()

for fold, (tr_idx, val_idx) in enumerate(cv.split(X.values, y_arr)):
    print(f'--- Fold {fold+1}/{N_FOLDS} ---')
    y_tr  = y_arr[tr_idx]
    y_val = y_arr[val_idx]
    X_tr_df  = X.iloc[tr_idx].copy()
    X_val_df = X.iloc[val_idx].copy()
    X_te_df  = test_X.copy()

    if USE_CUML:
        for c in BASE_LE_COLS:
            for stat in TE_STATS_BASE:
                n  = f'TE_{c}_{stat}'
                TE = cumlTE(n_folds=10, smooth=4, split_method='random', stat=stat)
                X_tr_df[n]  = TE.fit_transform(X_tr_df[[c]], y_tr).astype('float32')
                X_val_df[n] = TE.transform(X_val_df[[c]]).astype('float32')
                X_te_df[n]  = TE.transform(X_te_df[[c]]).astype('float32')
        for c in TRIGRAM_LE_COLS:
            n  = f'TE_{c}_mean'
            TE = cumlTE(n_folds=10, smooth=4, split_method='random', stat='mean')
            X_tr_df[n]  = TE.fit_transform(X_tr_df[[c]], y_tr).astype('float32')
            X_val_df[n] = TE.transform(X_val_df[[c]]).astype('float32')
            X_te_df[n]  = TE.transform(X_te_df[[c]]).astype('float32')
        print(f'  TE done: {len(BASE_LE_COLS)*len(TE_STATS_BASE)+len(TRIGRAM_LE_COLS)} features')

    X_tr_df.drop(columns=ALL_LE_COLS, inplace=True)
    X_val_df.drop(columns=ALL_LE_COLS, inplace=True)
    X_te_df.drop(columns=ALL_LE_COLS, inplace=True)
    X_tr_df.fillna(0, inplace=True)
    X_val_df.fillna(0, inplace=True)
    X_te_df.fillna(0, inplace=True)
    X_tr  = X_tr_df.values.astype(np.float32)
    X_val = X_val_df.values.astype(np.float32)
    X_te  = X_te_df.values.astype(np.float32)

    # XGBoost
    xgb_m = XGBClassifier(**best_xgb_params, scale_pos_weight=scale_pos_weight)
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof_p  = xgb_m.predict_proba(X_val)[:, 1]
    xgb_test_p = xgb_m.predict_proba(X_te)[:, 1]
    xgb_auc    = roc_auc_score(y_val, xgb_oof_p)
    pl_mask = (xgb_test_p > TRES) | (xgb_test_p < 1 - TRES)
    if pl_mask.sum() > 0:
        X_pl = np.vstack([X_tr, X_te[pl_mask]])
        y_pl = np.concatenate([y_tr, (xgb_test_p[pl_mask] > 0.5).astype(int)])
        xgb_pl = XGBClassifier(**best_xgb_params, scale_pos_weight=scale_pos_weight)
        xgb_pl.fit(X_pl, y_pl, eval_set=[(X_val, y_val)], verbose=False)
        p2 = xgb_pl.predict_proba(X_val)[:, 1]
        if roc_auc_score(y_val, p2) > xgb_auc:
            xgb_oof_p, xgb_test_p, xgb_auc = p2, xgb_pl.predict_proba(X_te)[:,1], roc_auc_score(y_val, p2)
            pl_counts['XGB'] += 1
    oof_xgb[val_idx]  = xgb_oof_p
    test_xgb_oof     += xgb_test_p / N_FOLDS
    fold_scores['XGB'].append(xgb_auc)
    print(f'  XGB:  {xgb_auc:.5f}  iter={xgb_m.best_iteration}  pl={pl_mask.sum()}')

    # LightGBM
    lgbm_m = LGBMClassifier(
        **{**best_lgbm_params, 'n_estimators': 50000},
        objective='binary', metric='auc', device='cpu',
        n_jobs=-1, is_unbalance=True, random_state=SEED, verbose=-1
    )
    lgbm_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
               callbacks=[lgb_lib.early_stopping(ES_OTHER), lgb_lib.log_evaluation(-1)])
    lgbm_oof_p  = lgbm_m.predict_proba(X_val)[:, 1]
    lgbm_test_p = lgbm_m.predict_proba(X_te)[:, 1]
    lgbm_auc    = roc_auc_score(y_val, lgbm_oof_p)
    pl_mask = (lgbm_test_p > TRES) | (lgbm_test_p < 1 - TRES)
    if pl_mask.sum() > 0:
        X_pl = np.vstack([X_tr, X_te[pl_mask]])
        y_pl = np.concatenate([y_tr, (lgbm_test_p[pl_mask] > 0.5).astype(int)])
        lgbm_pl = LGBMClassifier(
            **{**best_lgbm_params, 'n_estimators': 50000},
            objective='binary', metric='auc', device='cpu',
            n_jobs=-1, is_unbalance=True, random_state=SEED, verbose=-1
        )
        lgbm_pl.fit(X_pl, y_pl, eval_set=[(X_val, y_val)],
                    callbacks=[lgb_lib.early_stopping(ES_OTHER), lgb_lib.log_evaluation(-1)])
        p2 = lgbm_pl.predict_proba(X_val)[:, 1]
        if roc_auc_score(y_val, p2) > lgbm_auc:
            lgbm_oof_p, lgbm_test_p, lgbm_auc = p2, lgbm_pl.predict_proba(X_te)[:,1], roc_auc_score(y_val, p2)
            pl_counts['LGBM'] += 1
    oof_lgbm[val_idx]  = lgbm_oof_p
    test_lgbm_oof     += lgbm_test_p / N_FOLDS
    fold_scores['LGBM'].append(lgbm_auc)
    print(f'  LGBM: {lgbm_auc:.5f}')

    # CatBoost
    cat_m = CatBoostClassifier(
        **{**best_cat_params, 'iterations': 5000},
        early_stopping_rounds=ES_OTHER, loss_function='Logloss', eval_metric='AUC',
        random_seed=SEED, verbose=0, task_type='CPU', auto_class_weights='Balanced'
    )
    cat_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
    cat_oof_p  = cat_m.predict_proba(X_val)[:, 1]
    cat_test_p = cat_m.predict_proba(X_te)[:, 1]
    cat_auc    = roc_auc_score(y_val, cat_oof_p)
    pl_mask = (cat_test_p > TRES) | (cat_test_p < 1 - TRES)
    if pl_mask.sum() > 0:
        X_pl = np.vstack([X_tr, X_te[pl_mask]])
        y_pl = np.concatenate([y_tr, (cat_test_p[pl_mask] > 0.5).astype(int)])
        cat_pl = CatBoostClassifier(
            **{**best_cat_params, 'iterations': 5000},
            early_stopping_rounds=ES_OTHER, loss_function='Logloss', eval_metric='AUC',
            random_seed=SEED, verbose=0, task_type='CPU', auto_class_weights='Balanced'
        )
        cat_pl.fit(X_pl, y_pl, eval_set=[(X_val, y_val)])
        p2 = cat_pl.predict_proba(X_val)[:, 1]
        if roc_auc_score(y_val, p2) > cat_auc:
            cat_oof_p, cat_test_p, cat_auc = p2, cat_pl.predict_proba(X_te)[:,1], roc_auc_score(y_val, p2)
            pl_counts['CAT'] += 1
    oof_cat[val_idx]  = cat_oof_p
    test_cat_oof     += cat_test_p / N_FOLDS
    fold_scores['CAT'].append(cat_auc)
    print(f'  CAT:  {cat_auc:.5f}  iter={cat_m.best_iteration_}')
    gc.collect()

xgb_cv_score  = roc_auc_score(y_arr, oof_xgb)
lgbm_cv_score = roc_auc_score(y_arr, oof_lgbm)
cat_cv_score  = roc_auc_score(y_arr, oof_cat)

print()
print('=' * 55)
print('OVERALL OOF ROC-AUC')
print('=' * 55)
print(f'XGB:  {xgb_cv_score:.5f}  (V11 SEED=42: 0.91796)')
print(f'LGBM: {lgbm_cv_score:.5f}  (V11 SEED=42: 0.91781)')
print(f'CAT:  {cat_cv_score:.5f}  (V11 SEED=42: 0.91767)')
print()
print('Pseudo label improvements:')
for model, cnt in pl_counts.items():
    print(f'  {model}: {cnt}/{N_FOLDS} folds')

## 7. Final Ensemble & Submissions

In [ ]:
assert not np.isnan(test_xgb_oof).any()
assert not np.isnan(test_lgbm_oof).any()
assert not np.isnan(test_cat_oof).any()
print('All sanity checks passed!')

def rank_blend(arrays, weights):
    n      = len(arrays[0])
    ranked = [rankdata(a) / n for a in arrays]
    return np.clip(sum(w * r for w, r in zip(weights, ranked)), 0, 1)

v35_cv = roc_auc_score(y_arr, rank_blend(
    [oof_xgb, oof_lgbm, oof_cat], weights=[0.40, 0.35, 0.25]
))

# V35: Rank Blend V12 (SEED=123)
blend_v35 = rank_blend(
    [test_xgb_oof, test_lgbm_oof, test_cat_oof],
    weights=[0.40, 0.35, 0.25]
)
sub_v35 = pd.DataFrame({'id': test['id'], 'Churn': blend_v35})
sub_v35.to_csv('submission_v35_rank_blend_v12.csv', index=False)
print(f'V35 (Rank Blend V12 SEED=123): CV={v35_cv:.5f}  mean={blend_v35.mean():.4f} -> saved!')

# V36: Cross V32 (BEST CV=0.91808) + V35
v32_preds = v32_sub.set_index('id').loc[test['id'], 'Churn'].values
blend_v36 = rank_blend([v32_preds, blend_v35], weights=[0.5, 0.5])
sub_v36 = pd.DataFrame({'id': test['id'], 'Churn': blend_v36})
sub_v36.to_csv('submission_v36_cross_v32_v35.csv', index=False)
print(f'V36 (V32+V35 Cross):           mean={blend_v36.mean():.4f} -> saved!')

# V37: 3-Way V32 + V35 + V29
v29_preds = v29_sub.set_index('id').loc[test['id'], 'Churn'].values
blend_v37 = rank_blend([v32_preds, blend_v35, v29_preds], weights=[0.40, 0.40, 0.20])
sub_v37 = pd.DataFrame({'id': test['id'], 'Churn': blend_v37})
sub_v37.to_csv('submission_v37_3way_v32_v35_v29.csv', index=False)
print(f'V37 (V32+V35+V29 3-way):       mean={blend_v37.mean():.4f} -> saved!')

print()
print('Submit order: V35 first -> V36 -> V37')

## 8. CV-LB Tracking & Summary

In [ ]:
blend_cv = roc_auc_score(y_arr, rank_blend(
    [oof_xgb, oof_lgbm, oof_cat], weights=[0.40, 0.35, 0.25]
))

tracking = pd.DataFrame([
    {'Version':'V29', 'CV':0.91643, 'Public_LB':0.91408, 'Notes':'Rank Blend V10'},
    {'Version':'V31', 'CV':None,    'Public_LB':0.91425, 'Notes':'3-Way V10 (was best)'},
    {'Version':'V32', 'CV':0.91808, 'Public_LB':0.91572, 'Notes':'BEST LB ★ Rank Blend V11'},
    {'Version':'V35', 'CV':blend_cv,'Public_LB':None,    'Notes':'Rank Blend V12 SEED=123 <- submit 1st'},
    {'Version':'V36', 'CV':None,    'Public_LB':None,    'Notes':'V32+V35 Cross <- submit 2nd'},
    {'Version':'V37', 'CV':None,    'Public_LB':None,    'Notes':'V32+V35+V29 3-way <- submit 3rd'},
])
tracking['Gap'] = tracking['Public_LB'] - tracking['CV']
tracking['Health'] = tracking['Gap'].apply(
    lambda x: 'Good'    if pd.notna(x) and abs(x) < 0.003
    else ('Warning'     if pd.notna(x) and abs(x) < 0.005
    else ('Danger'      if pd.notna(x) else '—'))
)
print('=' * 80)
print('CV-LB TRACKING')
print('=' * 80)
print(tracking.to_string(index=False))
print('=' * 80)
print()
print(f'V12 SEED=123 OOF: XGB={xgb_cv_score:.5f}  LGBM={lgbm_cv_score:.5f}  CAT={cat_cv_score:.5f}')
print(f'V12 Blend V35 CV: {blend_cv:.5f}  (V11 SEED=42: 0.91808)')
print()
print('If V35 CV > 0.91643: add V35 to dataset')
print('Key: V36 cross-blend of two ~0.918 CV models should push LB > 0.91572')